In [5]:
import numpy as np
import pandas as pd
import yfinance as yf

# Commodity futures tickers on Yahoo Finance
tickers = {
    "corn": "ZC=F",
    "soybean": "ZS=F",
    "wheat": "ZW=F",
}

frames = []

for commodity, ticker in tickers.items():
    daily = yf.download(
        ticker,
        start="2010-01-01",
        end="2026-01-01",
        auto_adjust=False,
        progress=False,
    )

    if daily.empty:
        print(f"No data returned for {commodity} ({ticker}).")
        continue

    # Newer yfinance versions can return a MultiIndex for columns.
    if isinstance(daily.columns, pd.MultiIndex):
        daily.columns = daily.columns.get_level_values(0)

    price_col = "Adj Close" if "Adj Close" in daily.columns else "Close"
    if price_col not in daily.columns:
        print(f"No price column available for {commodity} ({ticker}).")
        continue

    monthly = daily[[price_col]].resample("ME").last().copy()
    monthly["return"] = np.log(monthly[price_col]).diff()
    monthly["commodity"] = commodity
    monthly = monthly.reset_index().rename(columns={"Date": "date", price_col: "price"})
    frames.append(monthly)

if not frames:
    raise RuntimeError(
        "No commodity data was downloaded. Check your internet connection or ticker symbols."
    )

futures_df = pd.concat(frames, ignore_index=True)
print(futures_df.head())



Price       date   price    return commodity
0     2010-01-31  356.50       NaN      corn
1     2010-02-28  378.00  0.058560      corn
2     2010-03-31  345.00 -0.091350      corn
3     2010-04-30  366.25  0.059772      corn
4     2010-05-31  359.00 -0.019994      corn
